# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdullahIdrees291/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Method choice and why

I will use **Logistic Regression** because this is a simple and interpretable classification model. The target will indicate whether `trend_pct` is positive or not. This method is suitable for testing whether the available content signals provide useful predictive information without adding unnecessary complexity.

The model will be compared with the Week-4 baseline using the same data and evaluation approach where applicable. The goal is to measure whether the model provides useful decision-support beyond the simple baseline rule.


In [11]:
!git clone https://github.com/AbdullahIdrees291/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 162 (delta 62), reused 95 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 1.96 MiB | 8.96 MiB/s, done.
Resolving deltas: 100% (62/62), done.


In [12]:
import os

os.chdir("/content/flyrank-ml-internship")

print("Current folder:", os.getcwd())
print("\nData files:")

for root, dirs, files in os.walk("data"):
    for file in files:
        print(os.path.join(root, file))

Current folder: /content/flyrank-ml-internship

Data files:
data/raw/content_refresh_anonymized.csv


In [13]:
import pandas as pd
import numpy as np

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [14]:
# Section 1: Method setup and target check

import pandas as pd
import numpy as np

# Confirm the dataset
print("Dataset shape:", df.shape)

# Create binary target:
# 1 = positive trend, 0 = non-positive trend
model_df = df.copy()
model_df["positive_trend"] = (model_df["trend_pct"] > 0).astype(int)

print("\nTarget distribution:")
print(model_df["positive_trend"].value_counts())

print("\nTarget proportions:")
print(model_df["positive_trend"].value_counts(normalize=True).round(3))

print("\nTarget column created:", "positive_trend")

Dataset shape: (30000, 44)

Target distribution:
positive_trend
0    23546
1     6454
Name: count, dtype: int64

Target proportions:
positive_trend
0    0.785
1    0.215
Name: proportion, dtype: float64

Target column created: positive_trend


## 2. Split design

I will use a stratified train/test split, with 80% of the data for training and 20% for testing. Stratification keeps the proportion of positive and non-positive trends similar in both sets. I will keep the test set separate until final evaluation so the model is compared on unseen data.


In [15]:
# Section 2: Train/test split

from sklearn.model_selection import train_test_split

# Features and target
X = model_df.drop(columns=["positive_trend", "trend_pct"])
y = model_df["positive_trend"]

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining target proportion:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest target proportion:")
print(y_test.value_counts(normalize=True).round(3))

Training rows: 24000
Test rows: 6000

Training target proportion:
positive_trend
0    0.785
1    0.215
Name: proportion, dtype: float64

Test target proportion:
positive_trend
0    0.785
1    0.215
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

I will train Logistic Regression on the training set and evaluate it on the held-out test set. The model will be compared with the Week-4 rule-based baseline using the same test rows and the same classification metric. Accuracy alone will not be treated as sufficient because the positive class is smaller than the non-positive class, so I will also report precision, recall, F1, and ROC-AUC.


In [16]:
# Section 3: Train Logistic Regression

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Remove identifiers and target/leakage-related outcome columns
drop_cols = [
    "positive_trend",
    "trend_pct",
    "trend_direction",
    "content_id",
    "client_id"
]

X_model = model_df.drop(columns=drop_cols)
y_model = model_df["positive_trend"]

# Use the SAME 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.20,
    random_state=42,
    stratify=y_model
)

# Identify numeric and categorical columns
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Logistic Regression model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Metrics
model_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

print("LOGISTIC REGRESSION RESULTS")
for metric, value in model_metrics.items():
    print(f"{metric}: {value:.4f}")

LOGISTIC REGRESSION RESULTS
Accuracy: 0.8222
Precision: 0.5582
Recall: 0.8327
F1: 0.6683
ROC-AUC: 0.9132


In [17]:
# Section 3: Week-4 baseline vs Logistic Regression

# Recreate the Week-4 baseline rule on the SAME test rows
baseline_test = df.loc[X_test.index].copy()

# Staleness score
baseline_test["stale_score"] = pd.cut(
    baseline_test["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=[0, 1, 2, 3]
).astype(float).fillna(0)

# Search-volume score
baseline_test["volume_score"] = pd.cut(
    baseline_test["search_volume"],
    bins=[-np.inf, 10, 100, 1000, np.inf],
    labels=[0, 1, 2, 3]
).astype(float).fillna(0)

# Week-4 baseline score
baseline_test["baseline_score"] = (
    baseline_test["stale_score"] +
    baseline_test["volume_score"]
)

# Convert baseline score into a binary prediction
# Positive prediction when baseline score >= 3
baseline_pred = (baseline_test["baseline_score"] >= 3).astype(int)

# Baseline metrics
baseline_metrics = {
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred, zero_division=0),
    "Recall": recall_score(y_test, baseline_pred, zero_division=0),
    "F1": f1_score(y_test, baseline_pred, zero_division=0)
}

# Comparison table
comparison = pd.DataFrame([
    {"Model": "Week-4 Baseline", **baseline_metrics},
    {"Model": "Logistic Regression", **model_metrics}
])

print("MODEL VS BASELINE")
display(comparison.round(4))

MODEL VS BASELINE


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Week-4 Baseline,0.7293,0.2569,0.1363,0.1781,NaN
1,Logistic Regression,0.8222,0.5582,0.8327,0.6683,0.9132


## 4. Errors and interpretation

The Logistic Regression model measured substantially better than the Week-4 baseline on the held-out test set. The largest improvement was in recall and F1, meaning the model identified more positive-trend cases while maintaining better precision than the simple rule.

The remaining errors are false positives and false negatives. False positives are content items predicted to have positive trends that actually had non-positive trends, while false negatives are positive-trend items that the model missed. These errors show that the available signals are useful but not perfect.

The model uses multiple content and performance signals rather than only staleness and search volume. These results are decision-support evidence, not proof that changing content will cause a positive trend.


In [18]:
# Section 4: Error analysis

from sklearn.metrics import confusion_matrix, classification_report

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("CONFUSION MATRIX")
print(cm)

# Error counts
tn, fp, fn, tp = cm.ravel()

print("\nERROR SUMMARY")
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

# Show a few examples of model errors
error_df = X_test.copy()
error_df["actual"] = y_test
error_df["predicted"] = y_pred
error_df["probability_positive"] = y_prob

false_positives = error_df[
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1)
].copy()

false_negatives = error_df[
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0)
].copy()

print("\nFalse positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nSample false positives:")
display(false_positives.head(5))

print("\nSample false negatives:")
display(false_negatives.head(5))

CONFUSION MATRIX
[[3858  851]
 [ 216 1075]]

ERROR SUMMARY
True negatives: 3858
False positives: 851
False negatives: 216
True positives: 1075

False positives: 851
False negatives: 216

Sample false positives:


,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,actual,predicted,probability_positive
18000,110.0,0.12,LOW,2.71,keyword article,informational,NaN,NaN,NaN,gpt-4o-mini,...,4.51,6.8,16.67,66.67,0.0,low,page_1,0,1,0.680365
7769,10.0,1.00,HIGH,0.00,keyword article,transactional,NaN,NaN,NaN,unknown,...,0.00,33.7,0.00,0.00,0.0,low,page_3_5,0,1,0.531431
14857,0.0,0.00,LOW,0.00,comparison article,informational,4455.0,31834.0,NaN,gemini-2.5-flash,...,1.01,7.5,0.00,16.67,0.0,low,page_1,0,1,0.505203
26661,0.0,0.00,LOW,0.00,comparison article,informational,2369.0,15943.0,google,gemini-3-flash-preview,...,0.12,20.8,0.00,50.00,0.0,moderate,page_3_5,0,1,0.681895
3093,40.0,0.64,MEDIUM,0.51,keyword article,transactional,2775.0,18088.0,google,gemini-3-flash-preview,...,0.00,50.8,0.00,0.00,0.0,low,deep,0,1,0.505668



Sample false negatives:


,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,actual,predicted,probability_positive
6488,30.0,0.06,LOW,0.07,keyword article,transactional,2968.0,20039.0,google,gemini-3-flash-preview,...,0.17,5.8,0.0,0.0,0.0,moderate,page_1,1,0,0.385709
29348,NaN,NaN,NaN,NaN,feedly article,NaN,925.0,6560.0,NaN,gpt-4o-mini,...,6.25,8.2,0.0,100.0,0.0,low,page_1,1,0,0.456425
6378,0.0,0.00,LOW,0.00,keyword article,informational,3000.0,21911.0,google,gemini-3-flash-preview,...,0.00,7.2,0.0,200.0,0.0,low,page_1,1,0,0.293490
26672,NaN,NaN,NaN,NaN,feedly article,NaN,1426.0,9624.0,NaN,gpt-5-mini,...,0.00,7.1,0.0,0.0,0.0,low,page_1,1,0,0.300681
11387,10.0,1.00,HIGH,0.04,keyword article,transactional,2823.0,20848.0,google,gemini-3-flash-preview,...,16.67,6.0,0.0,100.0,0.0,low,page_1,1,0,0.324542


### Error analysis

The confusion matrix shows 3,858 true negatives, 851 false positives, 216 false negatives, and 1,075 true positives.

The model has more false positives (851) than false negatives (216). This means the model is more likely to flag some non-positive-trend content as positive than to miss an actual positive-trend case.

The false-positive examples show that the model can predict positive trend even when some observable signals are weak or mixed. The false-negative examples show that some positive-trend content has relatively low predicted probability, so the model does not always detect positive cases.

Overall, the errors suggest that the model captures useful patterns but is not perfect. The results should be treated as **decision-support**, not as proof of future performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.